### import essential libraries

In [1]:
from operator import mod
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import RandomOverSampler
from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

### Load Data

In [2]:
mnist = fetch_openml('mnist_784')

/usr/local/lib/python3.9/dist-packages/sklearn/datasets/_openml.py:968: FutureWarning: The default value of `parser` will change from `'liac-arff'` to `'auto'` in 1.4. You can set `parser='auto'` to silence this warning. Therefore, an `ImportError` will be raised from 1.4 if the dataset is dense and pandas is not installed. Note that the pandas parser may return different data types. See the Notes Section in fetch_openml's API doc for details.
  warn(


In [3]:
X, y = mnist['data'], mnist['target']

### turning the problem to binary classsification

In [4]:
y = np.where(np.isin(y, ['0', '1', '2', '3', '4']), 0, 1)

### Preprocessing

In [5]:
# Normalize the data
X = X / 255.0

In [6]:
# Standardize the data
scaler = StandardScaler()
X = scaler.fit_transform(X)

### train- test split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Define imbalance ratios

In [8]:
imbalance_ratios = [(1, 99), (5, 95), (10, 90), (20, 80), (30, 70)]

### Define a function to imbalance the data

In [9]:
def create_imbalanced_dataset(X, y, ratio):
    unique_classes = np.unique(y)
    if len(unique_classes) < 2:
        raise ValueError('The dataset must contain at least two classes.')
    
    X_imbalanced, y_imbalanced = [], []
    for i, cls in enumerate(unique_classes):
        samples = X[y == cls]
        n_samples = min(max(int(len(X) * ratio[i] / (ratio[0] + ratio[1])), 1), len(samples))
        
        resampled_samples = resample(samples, replace=False, n_samples=n_samples, random_state=42)
        
        X_imbalanced.append(resampled_samples)
        y_imbalanced.extend([cls] * n_samples)
    
    X_imbalanced = np.vstack(X_imbalanced)
    y_imbalanced = np.array(y_imbalanced)
    
    return X_imbalanced, y_imbalanced

### for each ratio implement three ideas and evaluate them

In [10]:
for ratio in imbalance_ratios:
    
    # Create imbalanced training set
    X_train_imbalanced, y_train_imbalanced = create_imbalanced_dataset(X_train, y_train, ratio)


    # Train and evaluate logistic regression on imbalanced dataset
    clf_imbalanced = LogisticRegression(max_iter=1000)
    clf_imbalanced.fit(X_train_imbalanced, y_train_imbalanced)
    y_pred_imbalanced = clf_imbalanced.predict(X_test)
    

    # Apply RandomOverSampler to balance the dataset by oversampling the minority class
    random_over_sampler = RandomOverSampler(sampling_strategy='minority', random_state=42)
    X_train_balanced_with_ros, y_train_balanced_with_ros = random_over_sampler.fit_resample(X_train_imbalanced, y_train_imbalanced)

    # Train and evaluate logistic regression on balanced dataset (with randomm over sampling)
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_balanced_with_ros, y_train_balanced_with_ros)
    y_pred_balanced_with_ros = model.predict(X_test)


    # Create and train the logistic regression model with class weighting
    model_with_class_weighting = LogisticRegression(class_weight='balanced', random_state=42)
    model_with_class_weighting.fit(X_train_imbalanced, y_train_imbalanced)
    y_pred_balanced_with_class_weighting = model_with_class_weighting.predict(X_test)

    

    # Train and evaluate XGBClassifier with logistic regression as the base learner
    xgb_clf = XGBClassifier(booster='gblinear', objective='binary:logistic', n_estimators=100, learning_rate=0.1, random_state=42)
    xgb_clf.fit(X_train_imbalanced, y_train_imbalanced)
    y_pred_balanced_with_xgb = xgb_clf.predict(X_test)



    # Calculate performance metrics
    metrics_imbalanced = [accuracy_score(y_test, y_pred_imbalanced), precision_score(y_test, y_pred_imbalanced), recall_score(y_test, y_pred_imbalanced), f1_score(y_test, y_pred_imbalanced)]
    metrics_balanced_with_ros = [accuracy_score(y_test, y_pred_balanced_with_ros), precision_score(y_test, y_pred_balanced_with_ros), recall_score(y_test, y_pred_balanced_with_ros), f1_score(y_test, y_pred_balanced_with_ros)]
    metrics_balanced_with_class_weighting = [accuracy_score(y_test, y_pred_balanced_with_class_weighting), precision_score(y_test, y_pred_balanced_with_class_weighting), recall_score(y_test, y_pred_balanced_with_class_weighting), f1_score(y_test, y_pred_balanced_with_class_weighting)]
    metrics_balanced_with_xgb= [accuracy_score(y_test, y_pred_balanced_with_xgb), precision_score(y_test, y_pred_balanced_with_xgb), recall_score(y_test, y_pred_balanced_with_xgb), f1_score(y_test, y_pred_balanced_with_xgb)]

    print(f"Imbalance ratio: {ratio[0]}:{ratio[1]}")
    print("Imbalanced dataset metrics: Accuracy: {:.4f}, Precision: {:.4f}, Recall: {:.4f}, F1-score: {:.4f}".format(*metrics_imbalanced))
    print("Balanced dataset with random over sampling metrics: Accuracy: {:.4f}, Precision: {:.4f}, Recall: {:.4f}, F1-score: {:.4f}".format(*metrics_balanced_with_ros))
    print("Balanced dataset with class weighting metrics: Accuracy: {:.4f}, Precision: {:.4f}, Recall: {:.4f}, F1-score: {:.4f}".format(*metrics_balanced_with_class_weighting))
    print("Balanced dataset with xgb metrics: Accuracy: {:.4f}, Precision: {:.4f}, Recall: {:.4f}, F1-score: {:.4f}".format(*metrics_balanced_with_xgb))
    print("\n")

/usr/local/lib/python3.9/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Imbalance ratio: 1:99
Imbalanced dataset metrics: Accuracy: 0.6269, Precision: 0.5713, Recall: 0.9944, F1-score: 0.7257
Balanced dataset with random over sampling metrics: Accuracy: 0.8170, Precision: 0.7670, Recall: 0.9067, F1-score: 0.8310
Balanced dataset with class weighting metrics: Accuracy: 0.8194, Precision: 0.7710, Recall: 0.9049, F1-score: 0.8326
Balanced dataset with xgb metrics: Accuracy: 0.6209, Precision: 0.5672, Recall: 0.9963, F1-score: 0.7229




/usr/local/lib/python3.9/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Imbalance ratio: 5:95
Imbalanced dataset metrics: Accuracy: 0.7711, Precision: 0.6894, Recall: 0.9810, F1-score: 0.8097
Balanced dataset with random over sampling metrics: Accuracy: 0.8596, Precision: 0.8575, Recall: 0.8600, F1-score: 0.8587
Balanced dataset with class weighting metrics: Accuracy: 0.8624, Precision: 0.8615, Recall: 0.8611, F1-score: 0.8613
Balanced dataset with xgb metrics: Accuracy: 0.7729, Precision: 0.6909, Recall: 0.9817, F1-score: 0.8110




/usr/local/lib/python3.9/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Imbalance ratio: 10:90
Imbalanced dataset metrics: Accuracy: 0.8238, Precision: 0.7517, Recall: 0.9632, F1-score: 0.8444
Balanced dataset with random over sampling metrics: Accuracy: 0.8614, Precision: 0.8636, Recall: 0.8561, F1-score: 0.8598
Balanced dataset with class weighting metrics: Accuracy: 0.8647, Precision: 0.8687, Recall: 0.8570, F1-score: 0.8628
Balanced dataset with xgb metrics: Accuracy: 0.8279, Precision: 0.7555, Recall: 0.9656, F1-score: 0.8478




/usr/local/lib/python3.9/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Imbalance ratio: 20:80
Imbalanced dataset metrics: Accuracy: 0.8592, Precision: 0.8134, Recall: 0.9296, F1-score: 0.8676
Balanced dataset with random over sampling metrics: Accuracy: 0.8689, Precision: 0.8772, Recall: 0.8555, F1-score: 0.8662
Balanced dataset with class weighting metrics: Accuracy: 0.8687, Precision: 0.8775, Recall: 0.8548, F1-score: 0.8660
Balanced dataset with xgb metrics: Accuracy: 0.8624, Precision: 0.8175, Recall: 0.9306, F1-score: 0.8704




/usr/local/lib/python3.9/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Imbalance ratio: 30:70
Imbalanced dataset metrics: Accuracy: 0.8683, Precision: 0.8451, Recall: 0.8996, F1-score: 0.8715
Balanced dataset with random over sampling metrics: Accuracy: 0.8692, Precision: 0.8777, Recall: 0.8558, F1-score: 0.8666
Balanced dataset with class weighting metrics: Accuracy: 0.8703, Precision: 0.8794, Recall: 0.8561, F1-score: 0.8676
Balanced dataset with xgb metrics: Accuracy: 0.8688, Precision: 0.8460, Recall: 0.8994, F1-score: 0.8719


